## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# valence normalized
filtered['valence_norm'] = filtered['valence'] * 2 - 1 

# popularity logged
filtered['popularity_log'] = np.log1p(filtered['popularity'])

filtered_valences = filtered[['valence', 'valence_norm']]
display(filtered_valences .head(5))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))
for ax, col_name in zip(axes, filtered_valences .columns):
    filtered_valences[col_name].plot(kind='hist', ax=ax, title=col_name)

plt.tight_layout()
plt.show()

---
## Year Range Comparisons

In [ ]:
year_ranges = ['2021-2023', '2016-2023', '2013-2023', '2010-2023', '2000-2023']
years_min = [2021, 2016, 2013, 2010, 2000]
cols = ['popularity', 'valence']

# filtering year ranges
filtered_list = []

for i, year in enumerate(years_min):
    filtered_list.append(filtered[filtered['year'] >= year].copy().reset_index(drop=True))
    print(f'\n({year_ranges[i]})\tRecords: {len(filtered_list[i]):,}')
    display(filtered_list[i][cols].describe())

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))

for i, (f, yr) in enumerate(zip(filtered_list, year_ranges)):

    ax_hist = axes[i, 0]   # left column  → valence distribution
    ax_scat = axes[i, 1]   # right column → valence vs popularity

    # ── Valence distribution ──────────────────────────────────────────────────
    ax_hist.hist(f['valence'], bins=40, color='orchid',
                 edgecolor='white', alpha=0.85)
    ax_hist.axvline(f['valence'].mean(), color='darkviolet',
                    linestyle='--',
                    label=f'Mean: {f["valence"].mean():.2f}')
    ax_hist.set_title(f'Valence Distribution ({yr})', fontweight='bold')
    ax_hist.set_xlabel('Valence (0 = negative, 1 = positive)')
    ax_hist.set_ylabel('Track Count')
    ax_hist.legend()

    # Valence vs Popularity scatter
    ax_scat.scatter(f['valence'], f['popularity'],
                    alpha=0.12, s=5, color='orchid')

    # Trend line so the relationship is easier to read
    z = np.polyfit(f['valence'], f['popularity'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 1, 100)
    ax_scat.plot(x_line, p(x_line), color='darkviolet',
                 linewidth=1.5, linestyle='--',
                 label=f'r = {f["valence"].corr(f["popularity"]):.3f}')

    ax_scat.set_title(f'Valence vs Popularity ({yr})', fontweight='bold')
    ax_scat.set_xlabel('Valence')
    ax_scat.set_ylabel('Popularity')
    ax_scat.set_xlim(0, 1)
    ax_scat.legend()

plt.suptitle('Valence & Popularity by Year Range', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# log popularity check
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
bins = 20

for i, (f, yr) in enumerate(zip(filtered_list, year_ranges)):

    ax_pop = axes[i, 0]
    ax_pop_log = axes[i, 1]

    # Popularity distribution
    ax_pop.hist(f['popularity'], bins=bins, color='teal',
                 edgecolor='white', alpha=0.85)
    ax_pop.axvline(f['popularity'].mean(), color='darkslategray',
                    linestyle='--',
                    label=f'Mean: {f["popularity"].mean():.2f}')
    ax_pop.set_title(f'Popularity Distribution ({yr})', fontweight='bold')
    ax_pop.set_xlabel('Popularity (0 - 100)')
    ax_pop.set_ylabel('Track Count')
    ax_pop.legend()

    # Log Popularity
    ax_pop_log.hist(f['popularity_log'], bins=bins, color='teal',
                 edgecolor='white', alpha=0.85)
    ax_pop_log.axvline(f['popularity_log'].mean(), color='darkslategray',
                    linestyle='--',
                    label=f'Mean: {f["popularity_log"].mean():.2f}')
    ax_pop_log.set_title(f'Log Popularity Distribution ({yr})', fontweight='bold')
    ax_pop_log.set_ylabel('Track Count')
    ax_pop_log.legend()

plt.suptitle('Popularity vs Log Popularity by Year Range', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Removing Popularity Outliers

In [ ]:
iters = 1

fig, ax = plt.subplots(iters, 5, figsize=(20, 12))

for i in range(iters):

    lower_whiskers = []
    upper_whiskers = []

    for j, (f, yr) in enumerate(zip(filtered_list, year_ranges)):
        bp_dict = ax[i, j].boxplot(f['popularity_log'])
        ax[i, j].set_title(f'({yr})', fontweight='bold')

        # Whiskers come in pairs per column: [lower_whisker, upper_whisker]
        whiskers = [whisker.get_ydata() for whisker in bp_dict["whiskers"]]

        # Get the values where the whiskers end
        lower_whiskers.append(whiskers[0][1])
        upper_whiskers.append(whiskers[1][1])

    # removes outliers based on pop log
    for k in range(len(filtered_list)):
        # print(f'({year_ranges[k]})')

        # print('OLD -> ' + str(len(filtered_list[k])))
        filtered_list[k] = filtered_list[k][filtered_list[k]['popularity_log'] > lower_whiskers[k]]
        filtered_list[k] = filtered_list[k][filtered_list[k]['popularity_log'] < upper_whiskers[k]]
        filtered_list[k].reset_index(drop=True, inplace=True)
        # print('NEW -> ' + str(len(filtered_list[k])))
        # print()

plt.suptitle('Log Popularity Boxplots', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# check pop log stats
for i, f in enumerate(filtered_list):
    print(f'({year_ranges[i]})')
    display(f['popularity_log'].describe())
    print('Coefficient of Variation (std / mean): ' + str(f['popularity_log'].std() / f['popularity_log'].mean()))
    print()

---
## Save Filtered Songs

In [ ]:
# all data
segmented_records = 25000

for i, f in enumerate(filtered_list):
    path = f'data_filtered/songs_{year_ranges[i]}'
    Path(path).mkdir(parents=True, exist_ok=True)
    total_parts = ceil(len(f) / segmented_records)

    for i in range(total_parts):
        f[(segmented_records * i): (segmented_records * i + segmented_records)].to_csv(f'{path}/{i + 1}.csv', index_label='index')

In [ ]:
# data check
for year_range in year_ranges:
    check = pd.read_csv(f'data_filtered/songs_{year_range}/1.csv')
    display(check.head())